# Exploración inicial - Electric Vehicle Population Data

Este notebook realiza una exploración inicial del dataset antes de construir el pipeline Medallion.

El objetivo es revisar:
- Estructura y tipos de datos.
- Cantidad de registros.
- Valores nulos.
- Registros duplicados.
- Valores y rangos de las principales variables.
- Posibles reglas de limpieza para la capa Silver.

In [0]:
from pyspark.sql import functions as F

catalog = "ev_project"
source_path = f"/Volumes/{catalog}/landing/source_files"

display(dbutils.fs.ls(source_path))

## 1. Carga y estructura del dataset

Se carga el archivo original desde el Volume de Unity Catalog para revisar su estructura antes de definir las tablas del pipeline.

In [0]:
file_path = f"{source_path}/Electric_Vehicle_Population_Data.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
print(f"Filas: {df.count():,}")
print(f"Columnas: {len(df.columns)}")

### Observaciones sobre el esquema

El dataset contiene 289,564 registros y 16 columnas.

Spark infiere correctamente los tipos de varias variables numéricas. Sin embargo, algunos campos que representan identificadores o códigos, como Postal Code, DOL Vehicle ID y 2020 Census Tract, deberán revisarse antes de definir el esquema de la capa Silver.

## 2. Calidad de los datos

Se revisan valores faltantes, registros duplicados y la unicidad de los identificadores para identificar las reglas de limpieza que serán necesarias en la capa Silver.

Se revisan dos formas en las que pueden aparecer datos faltantes:

- **Valores nulos (`NULL`):** ausencia de un valor en la columna.
- **Cadenas vacías:** campos de texto que no son nulos, pero contienen `""` o únicamente espacios.

Ambos casos se revisan por separado, ya que Spark no considera una cadena vacía como un valor nulo.

In [0]:
from pyspark.sql import functions as F

null_counts = [
    (column, df.filter(F.col(column).isNull()).count())
    for column in df.columns
]

null_df = spark.createDataFrame(
    null_counts,
    ["column", "null_count"]
).withColumn(
    "null_percentage",
    F.round(F.col("null_count") / df.count() * 100, 2)
)

display(null_df.orderBy(F.desc("null_count")))

In [0]:
string_columns = [
    field.name
    for field in df.schema.fields
    if field.dataType.simpleString() == "string"
]

empty_counts = []

for column in string_columns:
    count = df.filter(
        F.col(column).isNotNull() &
        (F.trim(F.col(column)) == "")
    ).count()

    empty_counts.append((column, count))

empty_df = spark.createDataFrame(
    empty_counts,
    ["column", "empty_count"]
)

display(empty_df.orderBy(F.desc("empty_count")))

In [0]:
total_rows = df.count()
unique_rows = df.distinct().count()

print(f"Filas totales: {total_rows:,}")
print(f"Filas únicas: {unique_rows:,}")
print(f"Duplicados completos: {total_rows - unique_rows:,}")

In [0]:
duplicate_dol_ids = (
    df.groupBy("DOL Vehicle ID")
      .count()
      .filter(F.col("count") > 1)
      .orderBy(F.desc("count"))
)

display(duplicate_dol_ids)

In [0]:
df.select(
    F.countDistinct("VIN (1-10)").alias("unique_partial_vins"),
    F.countDistinct("DOL Vehicle ID").alias("unique_dol_ids"),
    F.countDistinct("Postal Code").alias("unique_postal_codes"),
    F.countDistinct("2020 Census Tract").alias("unique_census_tracts")
).show()

### Hallazgos iniciales de calidad

- No se encontraron registros duplicados, ni filas completas duplicadas ni valores repetidos de DOL Vehicle ID, por lo que se utilizará para validar la unicidad de los vehículos durante el procesamiento.
- VIN (1-10) no es un identificador único, ya que contiene únicamente los primeros 10 caracteres del VIN. Se encontraron 17,883 valores distintos, por lo que esta columna no se utilizará para eliminar duplicados.
- La cantidad de valores nulos es baja en relación con el total de registros.
- Legislative District presenta la mayor cantidad de valores nulos, con 727 registros (0.25%).
- Los demás campos presentan cantidades mínimas de valores nulos.
- No se encontraron cadenas vacías en las columnas de texto.
- Los valores nulos no serán eliminados de forma general. Su tratamiento se definirá según el significado de cada variable.

## 3. Análisis de variables principales

Se revisan las principales variables categóricas y numéricas para identificar sus valores, rangos y posibles casos que requieran tratamiento durante la transformación.

In [0]:
categorical_columns = [
    "State",
    "Electric Vehicle Type",
    "Clean Alternative Fuel Vehicle (CAFV) Eligibility"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    df.groupBy(column) \
      .count() \
      .orderBy(F.desc("count")) \
      .show(truncate=False)

In [0]:
df.select(
    F.min("Model Year").alias("min_model_year"),
    F.max("Model Year").alias("max_model_year"),
    F.min("Electric Range").alias("min_electric_range"),
    F.max("Electric Range").alias("max_electric_range"),
    F.avg("Electric Range").alias("avg_electric_range")
).show()

In [0]:
zero_range = df.filter(F.col("Electric Range") == 0).count()

print(f"Vehículos con Electric Range = 0: {zero_range:,}")
print(f"Porcentaje: {zero_range / total_rows * 100:.2f}%")

In [0]:
range_cafv_validation = (
    df.groupBy(
        "Clean Alternative Fuel Vehicle (CAFV) Eligibility",
        (F.col("Electric Range") == 0).alias("range_is_zero")
    )
    .count()
    .orderBy(
        "Clean Alternative Fuel Vehicle (CAFV) Eligibility",
        "range_is_zero"
    )
)

display(range_cafv_validation)

In [0]:
valid_range_df = df.filter(
    F.col("Electric Range").isNotNull() &
    (F.col("Electric Range") > 0)
)

valid_range_df.select(
    F.count("*").alias("vehicles_with_known_range"),
    F.round(F.avg("Electric Range"), 2).alias("avg_electric_range"),
    F.min("Electric Range").alias("min_electric_range"),
    F.max("Electric Range").alias("max_electric_range")
).show()

### Hallazgos sobre las variables principales

- La mayoría de los registros corresponden al estado de Washington, aunque existen registros asociados a otros estados. Estos registros se conservarán debido a que el dataset puede incluir vehículos registrados en Washington cuyos propietarios residen fuera del estado.
- Electric Vehicle Type contiene únicamente las categorías BEV y PHEV.
- CAFV Eligibility presenta tres categorías: elegible, no elegible y elegibilidad desconocida.
- El año de modelo se encuentra entre 1999 y 2027.

### Tratamiento de Electric Range

Se encontró que los 188,456 registros con Electric Range igual a 0 corresponden a vehículos cuya elegibilidad CAFV indica que el rango de batería no ha sido investigado.

Por esta razón, estos valores no se interpretarán como una autonomía real de cero. En la capa Silver serán tratados como valores desconocidos para evitar que afecten los cálculos de autonomía promedio.

Excluyendo los valores desconocidos, existen 101,098 vehículos con autonomía registrada, con un promedio de 107.12 y un rango entre 1 y 337.

También se identificaron 10 registros elegibles para CAFV con Electric Range nulo. Estos valores se mantendrán como nulos al no contar con información suficiente para determinar su autonomía.

## 4. Revisión de Vehicle Location

Se revisa el formato de Vehicle Location para determinar si las coordenadas geográficas pueden separarse en columnas de longitud y latitud durante la transformación.

In [0]:
df.select("Vehicle Location") \
  .filter(F.col("Vehicle Location").isNotNull()) \
  .show(20, truncate=False)

In [0]:
location_pattern = r"^POINT \((-?\d+\.?\d*) (-?\d+\.?\d*)\)$"

location_validation = df.select(
    F.count(
        F.when(F.col("Vehicle Location").isNotNull(), 1)
    ).alias("non_null_locations"),

    F.count(
        F.when(
            F.col("Vehicle Location").rlike(location_pattern),
            1
        )
    ).alias("valid_format_locations"),

    F.count(
        F.when(
            F.col("Vehicle Location").isNotNull() &
            ~F.col("Vehicle Location").rlike(location_pattern),
            1
        )
    ).alias("invalid_format_locations")
)

display(location_validation)

### Vehicle Location

La columna Vehicle Location utiliza el formato `POINT (longitude latitude)`. Todos los registros no nulos cumplen con este formato.

En la capa Silver se conservará el valor original y se crearán las columnas `longitude` y `latitude` como valores numéricos para facilitar el análisis geográfico.

## 5. Conclusiones

La exploración permitió identificar las principales reglas que deberán considerarse durante la construcción de la capa Silver:

- Utilizar DOL Vehicle ID para validar la unicidad de los registros.
- Conservar VIN (1-10) como atributo, pero no utilizarlo como identificador único.
- Revisar los tipos de Postal Code, DOL Vehicle ID y 2020 Census Tract, ya que representan códigos o identificadores.
- Mantener los valores nulos cuando no exista información suficiente para reemplazarlos.
- Tratar Electric Range igual a 0 como autonomía desconocida cuando la elegibilidad CAFV indique que el rango de batería no ha sido investigado.
- Conservar los registros asociados a estados distintos de Washington.
- Extraer longitude y latitude a partir de Vehicle Location.
- Conservar los años de modelo encontrados en la fuente, incluido 2027.

Estas reglas serán utilizadas posteriormente para definir las transformaciones de la capa Silver.